# HaemoLynx pipeline tutorial

This notebook runs the image-to-model pipeline **one stage at a time**, calling
the same functions `examples/resistance_network_pipeline.py` calls, in the same
order. Nothing here is a tutorial-only code path: what you run below is the
pipeline.

| Stage | Call | What it does |
|-------|------|--------------|
| 0 | `segment(settings)` | Pick the mask to analyse; run ilastik first if asked to |
| 1 | `skeletonise(settings, inputs)` | Load the volume, resolve its voxel size, skeletonise |
| 2 | `build_network(settings, volume, SCHEMA)` | Skeleton and vessel masks → graph |
| 3 | `assign_boundaries(settings, network)` | Where flow enters and leaves |
| 4 | `assign_diameters(...)` → `build_haemodynamic_model(...)` | Branch orders, diameters, resistances |
| 5 | `solve(settings, model, boundaries)` | Pressures, flows, equivalent resistance |
| 6 | `export_results(...)` | VTK, statistics, plots |

**Every stage takes the same `settings` dict**, loaded from a YAML config and
described by a schema, so changing the run means changing a value in that dict
— never editing a call. Each stage returns a small dataclass the next one takes:
`SegmentedInputs` → `SkeletonisedVolume` → `VesselNetwork` → `BoundaryNodes` →
`HaemodynamicModel` → `Solution`.

**Plots:** each stage writes its own PNGs to `settings["plot_dir"]`; the cells
below show them. Set `SHOW_STAGE_PLOTS = False` to skip the inline display.

**Source of truth:** edit **this notebook only**. [`pipeline_tutorial.py`](pipeline_tutorial.py)
is auto-generated (do not edit by hand). Regenerate it after notebook changes:

```bash
pytest tests/integration/test_pipeline_tutorial.py
```

## Prerequisites

```bash
pip install HaemoLynx
```

That is everything Stages 1–6 need: the next cell installs it for you if it is
missing, and the tutorial runs on a synthetic vessel volume it builds itself,
so no data download and no repository checkout are required.

Two optional extras:

- **Working from a clone?** `pip install -e ".[dev]"` from the repository root
  instead. The tutorial then picks up the real cropped nerve mask in
  `tests/data/` and the pipeline's own `examples/resistance_pipeline_config.yaml`
  rather than the defaults, and you get the per-step graph plots from
  `tutorials/tutorial_plots.py`.
- **[ilastik](https://www.ilastik.org/download/)** (a separate program) for
  Stage 0, only if you want to segment your own raw image.

In [ ]:
# `from __future__` has to be the first statement in the exported .py, so
# it leads the first code cell rather than the setup cell below.
from __future__ import annotations

# Install HaemoLynx if this kernel does not already have it. Nothing happens
# when it is present, so a clone or an editable install is left alone.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("haemolynx") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "HaemoLynx"], check=True)

import haemolynx

print(f"HaemoLynx {haemolynx.__version__} from {haemolynx.__file__}")

In [ ]:
import json
import os
import pickle
import sys
from pathlib import Path

import networkx as nx
import numpy as np


def _resolve_tutorial_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "pipeline_tutorial.ipynb").exists():
            return cwd
        if (cwd / "tutorials" / "pipeline_tutorial.ipynb").exists():
            return cwd / "tutorials"
        for parent in [cwd, *cwd.parents]:
            if (parent / "tutorials" / "pipeline_tutorial.ipynb").exists():
                return parent / "tutorials"
        return cwd


TUTORIAL_DIR = _resolve_tutorial_dir()


def _resolve_repo_root(tutorial_dir: Path) -> Path | None:
    """The checkout this notebook sits in, or None when pip-installed."""
    env_root = os.environ.get("HAEMOLYNX_REPO_ROOT")
    if env_root:
        return Path(env_root)
    candidates = [tutorial_dir, tutorial_dir.parent, Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for start in candidates:
        if (start / "src" / "haemolynx").is_dir() and (start / "examples").is_dir():
            return start
    return None


REPO_ROOT = _resolve_repo_root(TUTORIAL_DIR)
if REPO_ROOT is not None:
    # In a checkout, prefer the working tree over anything already installed.
    for p in (REPO_ROOT / "src", REPO_ROOT / "examples"):
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))

from haemolynx import graph, haemodynamics, io, preprocessing, statistics, visualization
from haemolynx.parsers import configure_console_logging

# The eight stages the example runs, in the order it runs them.
from haemolynx.pipeline import (
    assign_boundaries,
    assign_diameters,
    build_haemodynamic_model,
    build_network,
    default_schema,
    export_results,
    preflight,
    resolve_settings,
    segment,
    skeletonise,
    solve,
    write_default_config,
)

SCHEMA = default_schema()

# The stages report their progress through `logging`; send it to the notebook.
configure_console_logging()

# The plot helpers live beside the notebook in the repository. Without them the
# pipeline still runs; only the inline display of the saved figures is skipped.
if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))
try:
    from tutorial_plots import in_jupyter, show_stage_plots
except ModuleNotFoundError:
    print("tutorial_plots.py not found (installed, no checkout): figures are saved to disk, not shown inline.")

    def in_jupyter() -> bool:
        return False

    def show_stage_plots(stage_title, paths, *, enabled=True, **kwargs) -> None:
        if enabled:
            existing = [str(p) for p in paths if Path(p).exists()]
            print(f"{stage_title}: {', '.join(existing) if existing else 'no figures'}")

print(f"Repository root: {REPO_ROOT if REPO_ROOT else 'none - running from an installed HaemoLynx'}")
print(f"Running in Jupyter: {in_jupyter()}")

## Configuration: one settings dict

Every stage below reads this dict. It comes from a YAML config file validated
against `SCHEMA`, which is also what the CLI flags and a future GUI are built
from — so a setting has one name, one default, and one description wherever it
appears.

`TUTORIAL_OVERRIDES` is the only place this notebook differs from the shipped
configuration. To change what a stage does, change a value here (or edit the
YAML) — you never edit a call.

Run `python examples/resistance_network_pipeline.py --list-settings` to see all
140 of them, or `write_default_config("my_config.yaml")` to get a documented
file to edit.

In [ ]:
OUTPUT_DIR = Path(os.environ.get("HAEMOLYNX_TUTORIAL_OUTPUT_DIR", str(TUTORIAL_DIR / "outputs"))).resolve()
PLOT_DIR = Path(os.environ.get("HAEMOLYNX_TUTORIAL_PLOT_DIR", str(TUTORIAL_DIR / "plots"))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# In a checkout, the pipeline example's own config; otherwise the schema writes
# a default one, which is exactly what `write_default_config` gives any
# installed user.
CONFIG_PATH = REPO_ROOT / "examples" / "resistance_pipeline_config.yaml" if REPO_ROOT else None
if CONFIG_PATH is None or not CONFIG_PATH.exists():
    CONFIG_PATH = write_default_config(OUTPUT_DIR / "tutorial_config.yaml")

TUTORIAL_OVERRIDES = {
    # Where everything is written.
    "vtk_output_prefix": OUTPUT_DIR / "tutorial",
    "plot_dir": PLOT_DIR,
    # Meaning of each input array axis. Volumes load as canonical (z, y, x), so
    # this selects which axis is z -- the one projections look through.
    "image_axis_order": "zyx",
    # Skeletonisation: small values suit the small volume this tutorial uses.
    "skeleton_closing_radius": 1,
    "skeleton_bridge_gap_size": 1,
    "skeleton_min_branch_length": 3,
    "skeleton_max_bridge_distance": 2,
    "skeleton_component_connectivity": 3,
    "skeleton_min_component_percent": 1.0,
    # Graph topology repair.
    "graph_reconnect_threshold": 10.0,
    "final_orphan_reconnect_threshold": 3.0,
    "cluster_collapse_distance": 5.0,
    "min_stub_length": 3.0,
    # Boundary conditions: inlet and outlet pressures, in Pa.
    "inlet_p_bc": 1000.0,
    "outlet_p_bc": 500.0,
    # Inlet and outlet are picked by the box they fall in; the boxes are set in
    # Stage 1, once the volume's shape is known.
    "inlet_node_selection_method": "volume",
    "outlet_node_selection_method": "volume",
    "statistics": True,
    "statistics_mode": "fast",
    # Figures are saved and shown by the cells here, so the stages must not try
    # to open blocking windows of their own.
    "show_plots_in_ide": False,
    "hold_ide_plots_open": False,
    "interactive_plots": False,
    "final_render_mode": "2d",
}

# Display each stage's saved PNGs inline (notebook only).
SHOW_STAGE_PLOTS = True

print(f"Settings from: {CONFIG_PATH}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Plots: {PLOT_DIR}")
print(f"Inline stage plots enabled: {SHOW_STAGE_PLOTS and in_jupyter()}")

## Stage 0: Segment **your** raw image with ilastik

HaemoLynx Stages 1–6 need a **binary vessel mask** (foreground = vessel). This stage turns **your unsegmented** 3D TIFF into that mask using [ilastik](https://www.ilastik.org/) Pixel Classification.

> **Training happens in the ilastik GUI** (interactive). **Inference** runs headlessly inside `segment()` once you have a saved `.ilp` project — set `use_ilastik_segmentation` and the three paths below, and the stage does the rest.

### Step 0.1 — Install ilastik

1. Download ilastik from [ilastik.org/download](https://www.ilastik.org/download/) for your OS.
2. Note the executable path, e.g. Linux `~/ilastik-1.4.0-Linux/ilastik`, macOS `ilastik.app/Contents/MacOS/ilastik`, Windows `ilastik.exe`.
3. Set `ILASTIK_EXECUTABLE` in the code cell to that path (or add it to your `PATH`).

### Step 0.2 — Create a Pixel Classification project

1. Launch ilastik → **Create new project** → **Pixel Classification**.
2. **Input data:** Add your **raw** 3D volume (the same file you will set as `RAW_IMAGE_PATH`). Use TIFF or H5; shape should be `(Z, Y, X)` or `(Y, X, Z)` consistent with how you acquired the data.
3. **Features:** Default sigma features are usually fine for vessels; add more scales if capillaries are very thin or very thick.
4. **Labels:** Paint at least two classes on representative slices:
   - **Label 1** — vessel / capillary (foreground)
   - **Label 2** — background (or tissue you want excluded)
   Sample multiple Z positions and both bright/dim regions.
5. **Live update:** Enable live update and iterate labels until the preview segmentation looks acceptable on several slices.
6. **Train:** Click **Train** (or wait for live update) until the classifier stabilises.
7. **Save project:** **File → Save project as…** → e.g. `my_vessel_classifier.ilp`. This file is your **classifier** for headless runs.

### Step 0.3 — Check export settings (important)

In the **Prediction** / export section of the Pixel Classification workflow, ensure **Simple Segmentation** is available as an export source (this is what HaemoLynx requests headlessly). The saved `.ilp` embeds these settings.

### Step 0.4 — Run headless segmentation (code cell below)

1. Set `RAW_IMAGE_PATH` to your raw TIFF.
2. Set `ILASTIK_CLASSIFIER_PATH` to your `.ilp` file.
3. Set `RUN_STAGE_0_ILASTIK = True` and run the cell.

Output is written to `SEGMENTED_OUTPUT_PATH`. Then enable **your mask in Stage 1** (`USE_CUSTOM_SEGMENTED_IMAGE = True`) and run Stages 1–6.

### Skipping Stage 0 (tutorial demo only)

Leave `RUN_STAGE_0_ILASTIK = False` to skip segmentation and use the bundled pre-segmented `tests/data/Nerve_capillaries_cropped.tif` in Stage 1.

### Alternatives to ilastik

If you already have a mask from Fiji, napari, cellpose, etc., save it as a 3D TIFF and set `USE_CUSTOM_SEGMENTED_IMAGE = True` in Stage 1 with that path — you can skip Stage 0 entirely.

In [ ]:
# --- Stage 0: point these at your data, then set RUN_STAGE_0_ILASTIK = True ---
RUN_STAGE_0_ILASTIK = False  # True after you have a trained .ilp classifier

if RUN_STAGE_0_ILASTIK:
    TUTORIAL_OVERRIDES.update(
        {
            "use_ilastik_segmentation": True,
            "ilastik_unsegmented_image_path": Path("/path/to/your/raw_microscopy.tif"),
            "ilastik_classifier_path": Path("/path/to/your_pixel_classifier.ilp"),
            "ilastik_executable": os.environ.get("ILASTIK_EXECUTABLE", "ilastik"),
            "ilastik_output_dir": OUTPUT_DIR / "segmentation",
        }
    )
    print("Stage 0: segment() will run ilastik and analyse the mask it writes.")
else:
    print("Stage 0 skipped (RUN_STAGE_0_ILASTIK=False): Stage 1 analyses an existing mask.")

## Stage 1: `segment()` and `skeletonise()`

`segment()` settles **which** image is analysed — running ilastik when Stage 0
asked for it, otherwise passing the mask straight through — and returns a
`SegmentedInputs`. `skeletonise()` then loads that volume, resolves its voxel
size from the file metadata, and reduces the vessels to a one-voxel-wide
skeleton, returning a `SkeletonisedVolume`.

Stages 1–6 operate on a **binary segmentation**, not raw fluorescence. The mask
should be foreground voxels (e.g. 255) on background (0); HaemoLynx binarises
it on load.

- **In a checkout:** `tests/data/Nerve_capillaries_cropped.tif`, a small cropped mask.
- **Installed, with no data to hand:** a synthetic branching volume built below.
- **Your own:** set `input_path` in the overrides, or run Stage 0.

In [ ]:
# --- Which mask to analyse -------------------------------------------------
# In a checkout, the small cropped nerve mask committed for the tests. Without
# one, a synthetic volume built here -- so this notebook runs on a bare
# `pip install HaemoLynx`, with no data to download.
DEFAULT_SEGMENTED_TIFF = REPO_ROOT / "tests" / "data" / "Nerve_capillaries_cropped.tif" if REPO_ROOT else None


def build_synthetic_vessel_volume(path: Path) -> Path:
    """A branching phantom: one trunk, two branches, four free ends.

    Vessels are drawn as thick lines through a 64 x 64 x 64 volume, which is
    enough for every stage below to do something real -- skeletonise, build a
    graph with junctions and free ends, order the branches, and solve a flow.
    """
    import tifffile

    volume = np.zeros((64, 64, 64), dtype=np.uint8)

    def draw(start, end, radius=2):
        start, end = np.asarray(start, dtype=float), np.asarray(end, dtype=float)
        steps = int(np.linalg.norm(end - start)) * 4 + 1
        for t in np.linspace(0.0, 1.0, steps):
            z, y, x = np.round(start + t * (end - start)).astype(int)
            z0, z1 = max(0, z - radius), min(volume.shape[0], z + radius + 1)
            y0, y1 = max(0, y - radius), min(volume.shape[1], y + radius + 1)
            x0, x1 = max(0, x - radius), min(volume.shape[2], x + radius + 1)
            volume[z0:z1, y0:y1, x0:x1] = 255

    # Points are (z, y, x). The tree runs along y because the boxes below take
    # the inlet from the first 20% of y and the outlet from the last 20%: the
    # trunk starts inside the inlet band and every free end lands in the outlet
    # band.
    draw((32, 5, 32), (32, 30, 32))          # trunk
    draw((32, 30, 32), (32, 50, 14))         # two branches...
    draw((32, 30, 32), (32, 50, 50))
    draw((32, 50, 14), (18, 60, 8))          # ...each splitting once more
    draw((32, 50, 14), (46, 60, 8))
    draw((32, 50, 50), (18, 60, 56))
    draw((32, 50, 50), (46, 60, 56))

    path.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(path, volume)
    return path


env_input = os.environ.get("HAEMOLYNX_TUTORIAL_INPUT_TIFF")
if RUN_STAGE_0_ILASTIK:
    INPUT_TIFF = None  # segment() fills input_path in from the ilastik run
elif env_input:
    INPUT_TIFF = Path(env_input).expanduser().resolve()
elif DEFAULT_SEGMENTED_TIFF is not None and DEFAULT_SEGMENTED_TIFF.exists():
    INPUT_TIFF = DEFAULT_SEGMENTED_TIFF.resolve()
else:
    INPUT_TIFF = build_synthetic_vessel_volume(OUTPUT_DIR / "synthetic_vessels.tif")
    print("No segmented image to hand, so this run uses the synthetic volume above.")

if INPUT_TIFF is not None:
    TUTORIAL_OVERRIDES["input_path"] = INPUT_TIFF

# Every setting for this run, validated against the schema. `resolve_settings`
# also fills in the tables derived from other settings, such as the diameter
# for each branch order.
settings = resolve_settings(schema=SCHEMA, config_path=CONFIG_PATH, overrides=TUTORIAL_OVERRIDES)

# The pre-run checks the examples run for you: every path a stage will need,
# checked before any work starts.
report = preflight(settings, SCHEMA)
assert report.ok, "settings are not runnable; see the checklist above"

In [ ]:
inputs = segment(settings)
print(f"Stage 0/1 input: {inputs.image_path} ({inputs.input_format})")

volume = skeletonise(settings, inputs)
print(f"Image shape: {volume.image.shape[:3]}")
print(f"Voxel size (x, y, z): {volume.voxel_size_xyz}")
print(f"Array-axis spacing (z, y, x): {volume.voxel_size_zyx}")
print(f"Skeleton voxels: {int(volume.skeleton.sum())}")

# The inlet and outlet boxes are in physical (z, y, x) MICRONS, not voxel
# indices, so they are built from the volume's own extent. Inlet: the first 20%
# along y. Outlet: the last 20%.
extent_zyx = [
    (dimension - 1) * spacing
    for dimension, spacing in zip(volume.image.shape[:3], volume.voxel_size_zyx)
]
y_band = 0.2 * extent_zyx[1]
settings["inlet_node_volumes"] = [((0.0, 0.0, 0.0), (extent_zyx[0], y_band, extent_zyx[2]))]
settings["outlet_node_volumes"] = [
    ((0.0, extent_zyx[1] - y_band, 0.0), (extent_zyx[0], extent_zyx[1], extent_zyx[2]))
]
print(f"Inlet box (um):  {settings['inlet_node_volumes'][0]}")
print(f"Outlet box (um): {settings['outlet_node_volumes'][0]}")

# Both are written by skeletonise(): the skeleton as loaded, and after the
# cleaning pass that bridges gaps and drops the smallest components.
show_stage_plots(
    "Stage 1: Skeleton",
    [PLOT_DIR / "raw_skeleton.png", PLOT_DIR / "skeleton_projection.png"],
    enabled=SHOW_STAGE_PLOTS,
)

## Stage 2: `build_network()`

Turns the skeleton into an `nx.MultiGraph` and repairs its topology in eleven
steps — stitching loops, reconnecting broken segments, collapsing clusters of
nearby junctions, pruning stubs, and merging away degree-2 nodes so one vessel
is one edge rather than a chain of them. It also loads any large/small vessel
masks the settings name, and returns a `VesselNetwork`.

The stage saves a plot after each step to `settings["plot_dir"]`, so you can see
exactly what each one changed.

In [ ]:
network = build_network(settings, volume, SCHEMA)
G = network.graph
print(f"Final: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Free ends (degree 1): {sum(1 for _n, d in G.degree() if d == 1)}")
print(f"Junctions (degree >= 3): {sum(1 for _n, d in G.degree() if d >= 3)}")

# One overlay per topology step, in the order the steps ran.
step_plots = sorted(PLOT_DIR.glob("graph_after_*.png"))
print(f"Step overlays saved: {len(step_plots)}")
show_stage_plots("Stage 2: Graph topology steps", step_plots, enabled=SHOW_STAGE_PLOTS)

## Stage 3: `assign_boundaries()`

Chooses where flow enters and leaves, from the boxes set in Stage 1, and
returns a `BoundaryNodes`. Only degree-1 terminals are eligible: pinning a
pressure on an interior junction would make it inject or remove flow mid-network.

`inlet_node_selection_method` decides how they are picked, and each method
reads a different setting:

| Method | Reads |
|---|---|
| `"volume"` | `inlet_node_volumes` — corner pairs in (z, y, x) microns |
| `"coordinates"` | `inlet_node_coordinates` — each point snaps to the nearest terminal |
| `"edge_percent"` | `boundary_first_percent` / `boundary_last_percent` / `boundary_axis` — the first and last bands of the network along one axis |
| `"all_degree_1"` | every terminal in the graph |
| `"degree_1_from_inlet"` | `boundary_distance_from_inlet_node` — every terminal further than that from a inlet node |

`"edge_percent"` is the default, and the only one that asks nothing of the
dataset: it needs no coordinate, box or mask, so it has something to say about
an image nobody has looked at yet. This notebook overrides it with `"volume"`
because it knows where its own vessels are.

With segmented arteriole/venule masks, set `automated_vessel_assignment` instead
and the terminals come from anatomy rather than geometry.

In [ ]:
boundaries = assign_boundaries(settings, network)
print(f"Inlet nodes:  {boundaries.inlet_nodes}")
print(f"Outlet nodes: {boundaries.outlet_nodes}")
print(f"Node pair for the equivalent resistance: {boundaries.resistance_node_pair}")
assert boundaries.inlet_nodes and boundaries.outlet_nodes, (
    "no boundary nodes: widen the boxes above, or pick another selection method"
)

### Optional Stage 3A/3B: interactive tree, route, and bifurcation viewer

These cells are **optional** and are not part of the Stage 3 → Stage 4 pipeline. (Added here for specific use cases of the repo)

Run them after `assign_boundaries()` if you want to:
- browse individual connected trees/components;
- choose an inlet and outlet within a tree;
- view the full selected route in 3D;
- identify bifurcations on that route; and
- highlight vessel lines connected to those bifurcations.

The viewer stores its selection separately and does **not** replace the pipeline's `boundaries`.


In [ ]:
# run the below command if widgets aren't installed
%pip install --quiet ipywidgets anywidget plotly


In [ ]:
# Stage 3A viewer setup
# This cell is for exploration only. It does NOT replace assign_boundaries().

#Change these two viewever only values based on the data you have e.g. if boundary axis is x, and inlet side is high, we have a high x inlet and low x outlet. 
VIEWER_BOUNDARY_AXIS = "y" # x, y, or z
VIEWER_INLET_SIDE = "low" #high or low

AXIS_TO_INDEX = {"x": 0, "y": 1, "z": 2}

boundary_axis = str(VIEWER_BOUNDARY_AXIS).strip().lower()
inlet_side = str(VIEWER_INLET_SIDE).strip().lower()

if boundary_axis not in AXIS_TO_INDEX:
    raise ValueError(
        f"VIEWER_BOUNDARY_AXIS must be one of {tuple(AXIS_TO_INDEX)}, "
        f"not {VIEWER_BOUNDARY_AXIS!r}."
    )

if inlet_side not in {"high", "low"}:
    raise ValueError(
        f"VIEWER_INLET_SIDE must be 'high' or 'low', "
        f"not {VIEWER_INLET_SIDE!r}."
    )

axis_index = AXIS_TO_INDEX[boundary_axis]


def node_axis_value(node):
    """Return a node's selected physical coordinate from its (x, y, z) position."""
    position = G.nodes[node].get("pos")
    if position is None:
        raise RuntimeError(f"Node {node!r} has no 'pos' attribute.")

    position = np.asarray(position, dtype=float).reshape(-1)
    if position.size < 3 or not np.all(np.isfinite(position[:3])):
        raise RuntimeError(
            f"Node {node!r} has no valid physical (x, y, z) position."
        )

    return float(position[axis_index])


print(
    "Optional viewer setup ready: "
    f"{inlet_side}-{boundary_axis} inlet -> "
    f"{'low' if inlet_side == 'high' else 'high'}-{boundary_axis} outlet."
)
print("Pipeline boundaries are unchanged:", boundaries.resistance_node_pair)


In [ ]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

LENGTH_UNIT = "physical units"  # Change to "µm", "mm", etc. if you have

# Geometry helpers
def node_xyz(node):
    """Return a graph node position as physical (x, y, z)."""
    position = np.asarray(
        G.nodes[node].get("pos"),
        dtype=float,
    ).reshape(-1)

    if position.size < 3 or not np.all(np.isfinite(position[:3])):
        raise RuntimeError(
            f"Node {node!r} has no valid physical (x, y, z) position."
        )

    return position[:3]


def normalise_points(value):
    """Convert stored edge geometry to an N x 3 float array."""
    try:
        points = np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None

    if points.ndim != 2:
        return None

    if points.shape[1] >= 3:
        points = points[:, :3]
    elif points.shape[0] >= 3:
        points = points[:3, :].T
    else:
        return None

    if len(points) < 2 or not np.all(np.isfinite(points)):
        return None

    return points


def endpoint_fit_error(points_xyz, u_xyz, v_xyz):
    """Measure how well a centreline's ends match nodes u and v."""
    forward = (
        np.linalg.norm(points_xyz[0] - u_xyz)
        + np.linalg.norm(points_xyz[-1] - v_xyz)
    )
    reverse = (
        np.linalg.norm(points_xyz[0] - v_xyz)
        + np.linalg.norm(points_xyz[-1] - u_xyz)
    )
    return min(forward, reverse)


def edge_points_xyz(u, v, data):
    """Return an edge centreline in physical xyz coordinates."""
    u_xyz = node_xyz(u)
    v_xyz = node_xyz(v)

    geometry_keys = (
        "path_xyz",
        "points_xyz",
        "coordinates_xyz",
        "coords_xyz",
        "path",
        "points",
        "coordinates",
        "coords",
        "pixel_path",
        "voxel_path",
        "skeleton_path",
    )

    raw_points = None

    for geometry_key in geometry_keys:
        if geometry_key not in data:
            continue

        raw_points = normalise_points(data[geometry_key])
        if raw_points is not None:
            break

    if raw_points is None:
        return np.vstack([u_xyz, v_xyz])

    voxel_xyz = np.asarray(
        G.graph.get(
            "image_voxel_size_xyz",
            (1.0, 1.0, 1.0),
        ),
        dtype=float,
    )

    candidates = (
        raw_points,
        raw_points[:, ::-1],
        raw_points * voxel_xyz,
        raw_points[:, ::-1] * voxel_xyz,
    )

    return min(
        candidates,
        key=lambda points: endpoint_fit_error(
            points,
            u_xyz,
            v_xyz,
        ),
    )


def orient_points(points_xyz, u, v):
    """Orient an edge centreline from node u towards node v."""
    u_xyz = node_xyz(u)

    forward_error = np.linalg.norm(points_xyz[0] - u_xyz)
    reverse_error = np.linalg.norm(points_xyz[-1] - u_xyz)

    return points_xyz if forward_error <= reverse_error else points_xyz[::-1]


def append_polyline(target_x, target_y, target_z, points_xyz):
    """Append one polyline followed by a Plotly line break."""
    target_x.extend(points_xyz[:, 0].tolist())
    target_y.extend(points_xyz[:, 1].tolist())
    target_z.extend(points_xyz[:, 2].tolist())

    target_x.append(None)
    target_y.append(None)
    target_z.append(None)

# Edge lengths

def edge_physical_length(u, v, key, data):
    """Use a stored edge length, or calculate its centreline length."""
    possible_length_names = (
        "length",
        "physical_length",
        "path_length",
        "vessel_length",
        "branch_distance",
        "distance",
    )

    for name in possible_length_names:
        if name not in data:
            continue

        try:
            length = float(data[name])
        except (TypeError, ValueError):
            continue

        if np.isfinite(length) and length >= 0:
            return length

    points = edge_points_xyz(u, v, data)
    return float(
        np.linalg.norm(
            np.diff(points, axis=0),
            axis=1,
        ).sum()
    )


for u, v, key, data in G.edges(keys=True, data=True):
    data["_viewer_length"] = edge_physical_length(u, v, key, data)


def shortest_parallel_edge_key(graph_view, u, v):
    """Choose the shortest parallel edge between two path nodes."""
    return min(
        graph_view[u][v],
        key=lambda key: graph_view[u][v][key]["_viewer_length"],
    )

# Find every usable connected tree/component

def split_component_by_boundary_direction(component_nodes):
    """
    Apply the same Stage 3 terminal split independently to one component.

    Terminals are sorted on boundary_axis and split at the largest
    positive coordinate gap. inlet_side decides which side is the inlet.
    """
    terminals = [
        node
        for node in component_nodes
        if G.degree(node) == 1
    ]

    if len(terminals) < 2:
        return None

    terminals_sorted = sorted(
        terminals,
        key=lambda node: (node_axis_value(node), str(node)),
    )

    terminal_values = np.asarray(
        [node_axis_value(node) for node in terminals_sorted],
        dtype=float,
    )

    coordinate_min = float(terminal_values.min())
    coordinate_max = float(terminal_values.max())

    if np.isclose(coordinate_min, coordinate_max):
        return None

    coordinate_gaps = np.diff(terminal_values)
    positive_gap_indices = np.flatnonzero(coordinate_gaps > 1e-12)

    if positive_gap_indices.size == 0:
        return None

    midpoint_index = len(terminals_sorted) / 2.0

    split_after = max(
        positive_gap_indices,
        key=lambda index: (
            coordinate_gaps[index],
            -abs((index + 1) - midpoint_index),
        ),
    )

    split_index = int(split_after + 1)
    split_coordinate = float(
        (
            terminal_values[split_after]
            + terminal_values[split_after + 1]
        )
        / 2.0
    )

    low_side_nodes = terminals_sorted[:split_index]
    high_side_nodes = terminals_sorted[split_index:]

    if not low_side_nodes or not high_side_nodes:
        return None

    if inlet_side == "high":
        component_starting_nodes = high_side_nodes
        component_output_nodes = low_side_nodes
        default_pair = (
            max(component_starting_nodes, key=node_axis_value),
            min(component_output_nodes, key=node_axis_value),
        )
    else:
        component_starting_nodes = low_side_nodes
        component_output_nodes = high_side_nodes
        default_pair = (
            min(component_starting_nodes, key=node_axis_value),
            max(component_output_nodes, key=node_axis_value),
        )

    return {
        "nodes": set(component_nodes),
        "terminals": terminals_sorted,
        "low_side_nodes": low_side_nodes,
        "high_side_nodes": high_side_nodes,
        "starting_nodes": component_starting_nodes,
        "output_nodes": component_output_nodes,
        "default_pair": default_pair,
        "coordinate_min": coordinate_min,
        "coordinate_max": coordinate_max,
        "split_coordinate": split_coordinate,
    }


components = sorted(
    nx.connected_components(G),
    key=lambda component: (-len(component), min(map(str, component))),
)

all_tree_records = []

for component in components:
    record = split_component_by_boundary_direction(component)

    # Only components with at least two terminals and a usable high/low
    # coordinate split can be selected.
    if record is not None:
        all_tree_records.append(record)

if not all_tree_records:
    raise RuntimeError(
        "No connected component contains at least two degree-1 terminal "
        f"nodes that can be split along the {boundary_axis}-axis."
    )

for tree_index, record in enumerate(all_tree_records, start=1):
    record["tree_id"] = tree_index
    record["label"] = (
        f"Tree {tree_index} | {len(record['nodes'])} nodes | "
        f"{len(record['terminals'])} terminals | "
        f"{boundary_axis}={record['coordinate_min']:.3f} to "
        f"{record['coordinate_max']:.3f}"
    )

TREE_RECORDS_BY_ID = {
    record["tree_id"]: record
    for record in all_tree_records
}

print(
    f"Selectable trees/components: {len(all_tree_records)} "
    f"using {inlet_side}-{boundary_axis} inlets and "
    f"{'low' if inlet_side == 'high' else 'high'}-{boundary_axis} outlets."
)


# Selected-tree path and bifurcation helpers

def calculate_selected_path(tree_record, inlet_node, outlet_node):
    """Find the shortest physical route within the selected tree only."""
    if inlet_node not in tree_record["starting_nodes"]:
        raise ValueError("The selected inlet does not belong to this tree.")

    if outlet_node not in tree_record["output_nodes"]:
        raise ValueError("The selected outlet does not belong to this tree.")

    inlet_value = node_axis_value(inlet_node)
    outlet_value = node_axis_value(outlet_node)

    if inlet_side == "high" and inlet_value <= outlet_value:
        raise ValueError(
            f"The inlet must have a larger {boundary_axis} coordinate "
            "than the outlet."
        )

    if inlet_side == "low" and inlet_value >= outlet_value:
        raise ValueError(
            f"The inlet must have a smaller {boundary_axis} coordinate "
            "than the outlet."
        )

    tree_graph = G.subgraph(tree_record["nodes"])

    path_nodes = nx.shortest_path(
        tree_graph,
        source=inlet_node,
        target=outlet_node,
        weight="_viewer_length",
        method="dijkstra",
    )

    path_edges = []
    total_length = 0.0

    for u, v in zip(path_nodes[:-1], path_nodes[1:]):
        key = shortest_parallel_edge_key(tree_graph, u, v)
        length = float(tree_graph[u][v][key]["_viewer_length"])

        path_edges.append((u, v, key))
        total_length += length

    return path_nodes, path_edges, total_length


def selected_route_bifurcations(path_nodes):
    """Return graph junctions encountered by the selected route."""
    return [
        node
        for node in path_nodes
        if G.degree(node) >= 3
    ]


def edges_connected_to_nodes(tree_record, nodes):
    """Return all selected-tree edges incident to route bifurcations."""
    tree_graph = G.subgraph(tree_record["nodes"])
    connected_edges = []
    seen_edges = set()

    for bifurcation_node in nodes:
        for u, v, key, data in tree_graph.edges(
            bifurcation_node,
            keys=True,
            data=True,
        ):
            edge_identifier = (frozenset((u, v)), key)

            if edge_identifier in seen_edges:
                continue

            seen_edges.add(edge_identifier)
            connected_edges.append((u, v, key, data))

    return connected_edges


# Plot selected tree and route while retaining all trees

def make_full_path_figure(tree_record, inlet_node, outlet_node):
    path_nodes, path_edges, total_length = calculate_selected_path(
        tree_record,
        inlet_node,
        outlet_node,
    )

    route_bifurcation_nodes = selected_route_bifurcations(path_nodes)
    connected_bifurcation_edges = edges_connected_to_nodes(
        tree_record,
        route_bifurcation_nodes,
    )

    # All graph components/trees: faint grey.
    all_x, all_y, all_z = [], [], []

    for u, v, key, data in G.edges(keys=True, data=True):
        points = orient_points(edge_points_xyz(u, v, data), u, v)
        append_polyline(all_x, all_y, all_z, points)

    # The currently selected tree: darker grey.
    selected_tree_x, selected_tree_y, selected_tree_z = [], [], []
    selected_tree_graph = G.subgraph(tree_record["nodes"])

    for u, v, key, data in selected_tree_graph.edges(
        keys=True,
        data=True,
    ):
        points = orient_points(edge_points_xyz(u, v, data), u, v)
        append_polyline(
            selected_tree_x,
            selected_tree_y,
            selected_tree_z,
            points,
        )

    # All edges attached to bifurcations on the route: orange.
    connected_x, connected_y, connected_z = [], [], []

    for u, v, key, data in connected_bifurcation_edges:
        points = orient_points(edge_points_xyz(u, v, data), u, v)
        append_polyline(connected_x, connected_y, connected_z, points)

    # Selected inlet-to-outlet route: red.
    path_x, path_y, path_z = [], [], []

    for u, v, key in path_edges:
        data = G[u][v][key]
        points = orient_points(edge_points_xyz(u, v, data), u, v)
        append_polyline(path_x, path_y, path_z, points)

    inlet_positions = np.asarray(
        [node_xyz(node) for node in tree_record["starting_nodes"]],
        dtype=float,
    )
    outlet_positions = np.asarray(
        [node_xyz(node) for node in tree_record["output_nodes"]],
        dtype=float,
    )
    path_positions = np.asarray(
        [node_xyz(node) for node in path_nodes],
        dtype=float,
    )

    if route_bifurcation_nodes:
        bifurcation_positions = np.asarray(
            [node_xyz(node) for node in route_bifurcation_nodes],
            dtype=float,
        )
    else:
        bifurcation_positions = np.empty((0, 3), dtype=float)

    figure = go.Figure()

    figure.add_trace(
        go.Scatter3d(
            x=all_x,
            y=all_y,
            z=all_z,
            mode="lines",
            line={"width": 2, "color": "lightgrey"},
            opacity=0.12,
            hoverinfo="skip",
            name="All trees/components",
        )
    )

    figure.add_trace(
        go.Scatter3d(
            x=selected_tree_x,
            y=selected_tree_y,
            z=selected_tree_z,
            mode="lines",
            line={"width": 3, "color": "grey"},
            opacity=0.40,
            hoverinfo="skip",
            name=f"Selected tree {tree_record['tree_id']}",
        )
    )

    # Draw orange connected branches before the red route so the route
    # remains visibly red where the two overlap.
    if connected_bifurcation_edges:
        figure.add_trace(
            go.Scatter3d(
                x=connected_x,
                y=connected_y,
                z=connected_z,
                mode="lines",
                line={"width": 7, "color": "darkorange"},
                opacity=0.95,
                hoverinfo="skip",
                name="Lines connected to route bifurcations",
            )
        )

    figure.add_trace(
        go.Scatter3d(
            x=path_x,
            y=path_y,
            z=path_z,
            mode="lines",
            line={"width": 9, "color": "crimson"},
            hoverinfo="skip",
            name="Selected full path",
        )
    )

    figure.add_trace(
        go.Scatter3d(
            x=path_positions[:, 0],
            y=path_positions[:, 1],
            z=path_positions[:, 2],
            mode="markers",
            marker={"size": 4, "color": "crimson"},
            text=[
                f"Path node {node}; degree {G.degree(node)}"
                for node in path_nodes
            ],
            hovertemplate="%{text}<extra></extra>",
            name="Path nodes",
        )
    )

    if route_bifurcation_nodes:
        figure.add_trace(
            go.Scatter3d(
                x=bifurcation_positions[:, 0],
                y=bifurcation_positions[:, 1],
                z=bifurcation_positions[:, 2],
                mode="markers",
                marker={
                    "size": 5,
                    "color": "purple",
                    "symbol": "diamond",
                    "line": {"width": 1, "color": "black"},
                },
                text=[
                    (
                        f"Route bifurcation {node}<br>"
                        f"Degree: {G.degree(node)}<br>"
                        f"x={node_xyz(node)[0]:.3f}, "
                        f"y={node_xyz(node)[1]:.3f}, "
                        f"z={node_xyz(node)[2]:.3f}"
                    )
                    for node in route_bifurcation_nodes
                ],
                hovertemplate="%{text}<extra></extra>",
                name="Bifurcations on selected route",
            )
        )

    figure.add_trace(
        go.Scatter3d(
            x=inlet_positions[:, 0],
            y=inlet_positions[:, 1],
            z=inlet_positions[:, 2],
            mode="markers",
            marker={
                "size": 6,
                "color": "royalblue",
                "symbol": "diamond",
            },
            text=[
                (
                    f"Inlet node {node}<br>"
                    f"{boundary_axis}={node_axis_value(node):.3f}"
                )
                for node in tree_record["starting_nodes"]
            ],
            hovertemplate="%{text}<extra></extra>",
            name=f"{inlet_side}-{boundary_axis} inlets",
        )
    )

    outlet_side = "low" if inlet_side == "high" else "high"

    figure.add_trace(
        go.Scatter3d(
            x=outlet_positions[:, 0],
            y=outlet_positions[:, 1],
            z=outlet_positions[:, 2],
            mode="markers",
            marker={
                "size": 6,
                "color": "seagreen",
                "symbol": "square",
            },
            text=[
                (
                    f"Outlet node {node}<br>"
                    f"{boundary_axis}={node_axis_value(node):.3f}"
                )
                for node in tree_record["output_nodes"]
            ],
            hovertemplate="%{text}<extra></extra>",
            name=f"{outlet_side}-{boundary_axis} outlets",
        )
    )

    figure.update_layout(
        title=(
            f"Tree {tree_record['tree_id']}: inlet {inlet_node} → "
            f"outlet {outlet_node}<br>"
            f"Total centreline length: {total_length:.3f} {LENGTH_UNIT}; "
            f"{len(path_edges)} graph segments; "
            f"{len(route_bifurcation_nodes)} route bifurcations"
        ),
        height=750,
        margin={"l": 0, "r": 0, "t": 90, "b": 0},
        scene={
            "xaxis_title": f"x ({LENGTH_UNIT})",
            "yaxis_title": f"y ({LENGTH_UNIT})",
            "zaxis_title": f"z ({LENGTH_UNIT})",
            "aspectmode": "data",
        },
    )

    return (
        figure,
        path_nodes,
        path_edges,
        total_length,
        route_bifurcation_nodes,
        connected_bifurcation_edges,
    )


# Widgets

def inlet_options_for_tree(tree_record):
    reverse = inlet_side == "high"

    return [
        (
            f"Node {node} | {boundary_axis}={node_axis_value(node):.3f}",
            node,
        )
        for node in sorted(
            tree_record["starting_nodes"],
            key=node_axis_value,
            reverse=reverse,
        )
    ]


def outlet_options_for_tree(tree_record):
    reverse = inlet_side != "high"

    return [
        (
            f"Node {node} | {boundary_axis}={node_axis_value(node):.3f}",
            node,
        )
        for node in sorted(
            tree_record["output_nodes"],
            key=node_axis_value,
            reverse=reverse,
        )
    ]


first_tree = all_tree_records[0]

tree_selector = widgets.Dropdown(
    options=[
        (record["label"], record["tree_id"])
        for record in all_tree_records
    ],
    value=first_tree["tree_id"],
    description="Tree:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="470px"),
)

inlet_selector = widgets.Dropdown(
    options=inlet_options_for_tree(first_tree),
    value=first_tree["default_pair"][0],
    description=f"{inlet_side}-{boundary_axis} inlet:",
    style={"description_width": "initial"},
)

outlet_side = "low" if inlet_side == "high" else "high"

outlet_selector = widgets.Dropdown(
    options=outlet_options_for_tree(first_tree),
    value=first_tree["default_pair"][1],
    description=f"{outlet_side}-{boundary_axis} outlet:",
    style={"description_width": "initial"},
)

show_button = widgets.Button(
    description="Show selected route",
    icon="refresh",
    button_style="primary",
)

viewer_output = widgets.Output()
updating_controls = False


def store_viewer_selection(tree_record, inlet_node, outlet_node):
    """Store the current 3B choice without changing Stage 3 flow boundaries."""
    global selected_tree_record
    global selected_tree_nodes
    global selected_tree_starting_nodes
    global selected_tree_output_nodes
    global selected_resistance_node_pair

    selected_tree_record = tree_record
    selected_tree_nodes = set(tree_record["nodes"])
    selected_tree_starting_nodes = list(tree_record["starting_nodes"])
    selected_tree_output_nodes = list(tree_record["output_nodes"])
    selected_resistance_node_pair = (inlet_node, outlet_node)


def update_full_path(*_):
    if updating_controls:
        return

    tree_record = TREE_RECORDS_BY_ID[tree_selector.value]
    inlet_node = inlet_selector.value
    outlet_node = outlet_selector.value

    store_viewer_selection(tree_record, inlet_node, outlet_node)

    with viewer_output:
        viewer_output.clear_output(wait=True)

        try:
            (
                figure,
                path_nodes,
                path_edges,
                total_length,
                route_bifurcation_nodes,
                connected_bifurcation_edges,
            ) = make_full_path_figure(
                tree_record,
                inlet_node,
                outlet_node,
            )
        except (
            nx.NetworkXNoPath,
            nx.NodeNotFound,
            ValueError,
        ) as error:
            print(error)
            return

        print(
            f"Tree {tree_record['tree_id']}: "
            f"{len(tree_record['nodes'])} nodes, "
            f"{len(tree_record['terminals'])} terminals."
        )
        print(
            f"Tree-specific {boundary_axis} split: "
            f"{tree_record['split_coordinate']:.3f}"
        )
        print(
            f"Selected full path length: {total_length:.6f} {LENGTH_UNIT}; "
            f"{len(path_edges)} segments; {len(path_nodes)} nodes."
        )
        print(
            f"Bifurcations on selected route: "
            f"{len(route_bifurcation_nodes)}"
        )
        print(
            f"Connected vessel lines highlighted: "
            f"{len(connected_bifurcation_edges)}"
        )
        print(
            f"Viewer selection stored as selected_resistance_node_pair: "
            f"{selected_resistance_node_pair}"
        )

        display(figure)


def on_tree_change(change):
    global updating_controls

    if change.get("name") != "value" or change.get("new") is None:
        return

    tree_record = TREE_RECORDS_BY_ID[change["new"]]

    updating_controls = True
    try:
        inlet_selector.options = inlet_options_for_tree(tree_record)
        outlet_selector.options = outlet_options_for_tree(tree_record)
        inlet_selector.value = tree_record["default_pair"][0]
        outlet_selector.value = tree_record["default_pair"][1]
    finally:
        updating_controls = False

    store_viewer_selection(
        tree_record,
        inlet_selector.value,
        outlet_selector.value,
    )
    update_full_path()


tree_selector.observe(on_tree_change, names="value")
inlet_selector.observe(update_full_path, names="value")
outlet_selector.observe(update_full_path, names="value")
show_button.on_click(update_full_path)

controls_row_1 = widgets.HBox([tree_selector, show_button])
controls_row_2 = widgets.HBox([inlet_selector, outlet_selector])

display(
    widgets.VBox([controls_row_1, controls_row_2]),
    viewer_output,
)

# Click "Show selected route" or change a dropdown to view.

## Stage 4: `assign_diameters()` and `build_haemodynamic_model()`

`assign_diameters()` walks out from the inlets to give every edge a
**branch order**, then uses that order as the key into the diameter table
(`diameter_by_branch_order`, derived from the settings). With
`use_fwhm_edge_diameters` on it measures each vessel from the raw image instead
and the table is only a fallback.

`build_haemodynamic_model()` turns diameter and length into a **resistance** for
each edge, using the Poiseuille law with a diameter-dependent apparent
viscosity, and writes `resistance` (Pa·s/m³) and `conductance` (m³/(Pa·s))
together. It returns a `HaemodynamicModel`.

> Vessels between 7 µm and 100 µm raise a `PlaceholderViscosityWarning`: the
> viscosity there is held constant rather than transitioning smoothly, so treat
> those resistances as order-of-magnitude (issue #90).

In [ ]:
diameters = assign_diameters(settings, network, boundaries, SCHEMA)
model = build_haemodynamic_model(settings, diameters)
G = model.graph

# Edges the walk could not reach from an inlet keep no branch order, and are
# left without a resistance rather than being given a made-up one.
orders = sorted({data["branch_order"] for _u, _v, data in G.edges(data=True) if "branch_order" in data})
print(f"Branch orders assigned: {orders[:10]}{' ...' if len(orders) > 10 else ''}")
resistances = [data["resistance"] for _u, _v, data in G.edges(data=True) if "resistance" in data]
print(f"Edges with a resistance: {len(resistances)} of {G.number_of_edges()}")
if resistances:
    print(f"Resistance range (Pa.s/m^3): {min(resistances):.3e} to {max(resistances):.3e}")

# The branch-order figure is drawn by Stage 6, once flows are on the graph too.

## Stage 5: `solve()`

Builds the conductance matrix from the edge conductances, pins `inlet_p_bc` at
the inlets and `outlet_p_bc` at the outlets, and solves for the pressure at
every node — then writes the resulting flow onto each edge. With
`do_equiv_resistance_calculation` on it also computes the two-point resistance
between the boundary pair, which is the whole network reduced to a single
number. It returns a `Solution`.

In [ ]:
solution = solve(settings, model, boundaries)

pressures = solution.pressure
print(f"Solved for {len(solution.node_list)} nodes")
print(f"Pressure range (Pa): {float(np.min(pressures)):.1f} to {float(np.max(pressures)):.1f}")
print(f"Equivalent resistance (Pa.s/m^3): {solution.equivalent_resistance:.6e}")

flows = [abs(data["flow_signed"]) for _u, _v, data in model.graph.edges(data=True) if "flow_signed" in data]
print(f"Edge flows (m^3/s): {min(flows):.3e} to {max(flows):.3e}")

## Stage 6: `export_results()`

The last stage writes everything out: the VTK files (vessels, nodes and
pericytes, carrying resistance, pressure and flow as cell arrays), the network
statistics CSV, and the final plots.

In [ ]:
export_results(settings, network, model, solution)

written = sorted(OUTPUT_DIR.glob("*.vtp")) + sorted(OUTPUT_DIR.glob("*.csv"))
for path in written:
    print(f"  {path.name}  ({path.stat().st_size:,} bytes)")

stats_csv = next((p for p in written if p.suffix == ".csv"), None)
if stats_csv is not None:
    print()
    print(stats_csv.read_text(encoding="utf-8").splitlines()[0])

show_stage_plots(
    "Stage 6: Final network",
    [
        PLOT_DIR / "geometry_with_branch_orders.png",
        PLOT_DIR / "edges_and_nodes_overlay.png",
        PLOT_DIR / "node_degree_distribution.png",
        PLOT_DIR / "final_graph.png",
    ],
    enabled=SHOW_STAGE_PLOTS,
)

### View VTK outputs in ParaView

After the cell above finishes, open the exported `.vtp` files in [ParaView](https://www.paraview.org/) or similar VTK software (e.g. napari with a VTK plugin, [3D Slicer](https://www.slicer.org/)):

- ``stem`_tutorial_vessels.vtp` — vessel centreline geometry and attributes
- the vessels file also carries **flow** scalars once the solve has run
- ``stem`_tutorial_nodes.vtp` — graph nodes
- ``stem`_tutorial_pericytes.vtp` — pericyte points (if present)

In ParaView: **File → Open**, select the `.vtp` files, click **Apply**, then colour by array names such as `flow` or `branch_order`.

## Running all six at once

The example is exactly the cells above, one after the other:

```python
inputs     = segment(settings)
volume     = skeletonise(settings, inputs)
network    = build_network(settings, volume, SCHEMA)
boundaries = assign_boundaries(settings, network)
diameters  = assign_diameters(settings, network, boundaries, SCHEMA)
model      = build_haemodynamic_model(settings, diameters)
solution   = solve(settings, model, boundaries)
export_results(settings, network, model, solution)
```

`haemolynx.pipeline.run_pipeline_stages(settings, SCHEMA)` runs that sequence
for you, and `python examples/resistance_network_pipeline.py` runs it from the
command line against `examples/resistance_pipeline_config.yaml`.

## Adapting for your own data

1. **Edit this notebook** (`pipeline_tutorial.ipynb`) — not the generated `.py`.
2. **Your own mask:** put its path in `TUTORIAL_OVERRIDES["input_path"]`.
3. **Your own raw image:** set `RUN_STAGE_0_ILASTIK = True` in Stage 0 and fill in
   the three ilastik paths; `segment()` runs the classifier and analyses what it writes.
4. **Boundaries:** change `inlet_node_selection_method` and the setting it
   reads (see the table in Stage 3), or set `automated_vessel_assignment` to take
   them from arteriole/venule masks.
5. **Tuning:** every skeleton and graph threshold is a key in `TUTORIAL_OVERRIDES`.
   `write_default_config("my_config.yaml")` writes all 140 settings with their
   documentation, ranges and defaults; edit that and pass it as `CONFIG_PATH`.
6. **Regenerate** `pipeline_tutorial.py`: `pytest tests/integration/test_pipeline_tutorial.py`.